# ML Assignment 2 — Classification Models

**Dataset:** Breast Cancer Wisconsin (Diagnostic)

This notebook implements the five classification models explicitly named in the assignment and evaluates them using six required metrics.

## 1. Import Libraries

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import time

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    accuracy_score, roc_auc_score, precision_score,
    recall_score, f1_score, matthews_corrcoef,
    confusion_matrix, classification_report
)

C:\Users\saiak\anaconda3\lib\site-packages\scipy\__init__.py:159: UserWarning: A NumPy version >=1.21.6 and <1.28.0 is required for this version of SciPy (detected version 1.21.5)
  warnings.warn(f"A NumPy version >={np_minversion} and <{np_maxversion}"


ValueError: numpy.ndarray size changed, may indicate binary incompatibility. Expected 96 from C header, got 88 from PyObject

## 2. Load and Inspect the Dataset

In [ ]:
data = load_breast_cancer(as_frame=True)
df = data.frame.copy()

print("Dataset shape:", df.shape)
print("\nFirst five rows:")
display(df.head())

print("\nClass distribution:")
display(df["target"].value_counts())

print("\nTarget mapping:")
print("0 =", data.target_names[0])
print("1 =", data.target_names[1])

### Dataset suitability
The dataset contains 569 instances and 30 numerical features, satisfying the assignment minimum of 500 instances and 12 features.

In [ ]:
X = df.drop(columns=["target"])
y = df["target"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    stratify=y,
    random_state=42
)

print("Training instances:", X_train.shape[0])
print("Testing instances:", X_test.shape[0])
print("Number of features:", X_train.shape[1])

## 3. Define the Classification Models

StandardScaler is used for Logistic Regression and kNN because these models are distance/scale sensitive.

In [ ]:
models = {
    "Logistic Regression": Pipeline([
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(max_iter=5000, random_state=42))
    ]),
    "Decision Tree": DecisionTreeClassifier(random_state=42),
    "kNN": Pipeline([
        ("scaler", StandardScaler()),
        ("model", KNeighborsClassifier(n_neighbors=5))
    ]),
    "Naive Bayes": GaussianNB(),
    "Random Forest": RandomForestClassifier(
        n_estimators=200,
        random_state=42
    )
}

print("Models:")
for name in models:
    print("-", name)

## 4. Train and Evaluate Models

In [ ]:
results = []
predictions = {}
probabilities = {}

for name, model in models.items():
    start = time.time()
    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]

    predictions[name] = y_pred
    probabilities[name] = y_prob

    results.append({
        "ML Model Name": name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "AUC": roc_auc_score(y_test, y_prob),
        "Precision": precision_score(y_test, y_pred),
        "Recall": recall_score(y_test, y_pred),
        "F1": f1_score(y_test, y_pred),
        "MCC": matthews_corrcoef(y_test, y_pred),
        "Time (sec)": time.time() - start
    })

results_df = pd.DataFrame(results)
display(results_df.round(4))

## 5. Confusion Matrices

In [ ]:
for name, y_pred in predictions.items():
    cm = confusion_matrix(y_test, y_pred)
    print(f"\n{name}")
    display(pd.DataFrame(
        cm,
        index=["Actual Malignant", "Actual Benign"],
        columns=["Predicted Malignant", "Predicted Benign"]
    ))

    plt.figure(figsize=(4, 3))
    sns.heatmap(cm, annot=True, fmt="d", cbar=False)
    plt.title(f"{name} — Confusion Matrix")
    plt.xlabel("Predicted")
    plt.ylabel("Actual")
    plt.show()

## 6. Classification Reports

In [ ]:
for name, y_pred in predictions.items():
    print("=" * 70)
    print(name)
    print("=" * 70)
    print(classification_report(
        y_test,
        y_pred,
        target_names=["Malignant", "Benign"],
        zero_division=0
    ))

## 7. Final Comparison

Logistic Regression is the overall winner for this split because it has the highest values across all six required metrics.

In [ ]:
display(results_df.drop(columns=["Time (sec)"]).round(4))

winner = results_df.loc[
    results_df["F1"].idxmax(), "ML Model Name"
]
print("Overall Winner:", winner)

## 8. Export Test Data

The test set is exported for use in the Streamlit application.

In [ ]:
test_data = X_test.copy()
test_data["target"] = y_test
test_data.to_csv("test_data.csv", index=False)
print("test_data.csv created with shape:", test_data.shape)